# 0. Import all core dependencies

In [1]:
# PyTorch - TODO
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from timm.models.swin_transformer import SwinTransformerBlock
import timm

# Optimisers and Schedulers - TODO
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

# Data Handling - TODO
from PIL import Image
import numpy as np
import pandas as pd
import cv2

# Data Visualisation - TODO
import matplotlib.pyplot as plt
import seaborn as sns

# Class Imbalance Handling - TODO
from imblearn.over_sampling import SMOTE
from collections import Counter

# Evaluation - TODO
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    recall_score
)

# Explainability - TODO
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Utilities
from tqdm import tqdm
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
import random
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Configure PyTorch to use my GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

C:\Users\aria_\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using cuda


# 1. Define all modules in Swin-Xception

# 2. Image Preprocessing function

In [11]:
NOTEBOOK_DIR = os.getcwd()

SRC_PATH = os.path.abspath("../../src")

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from datasets import FERDataset
from torchvision.transforms import v2

transform_train = v2.Compose([
    v2.Resize(size=(224, 224), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness = 0.2, contrast = 0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = v2.Compose([
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

fer_train = FERDataset(os.path.abspath("../../datasets/FER2013/train"), transform_train)
fer_test = FERDataset(os.path.abspath("../../datasets/FER2013/test"), transform_test)
raf_test = FERDataset(os.path.abspath("../../datasets/RAFDB/DATASET/test"), transform_test)

train_loader = DataLoader(fer_train, batch_size=32, shuffle=True, num_workers=4)
test_fer_loader = DataLoader(fer_test, batch_size=32, shuffle=False, num_workers=4)
test_raf_loader = DataLoader(raf_test, batch_size=32,shuffle=False, num_workers=4)

print(f"FER2013 Training set images: {len(fer_train)}")
print(f"FER2013 Test set images: {len(fer_test)}")
print(f"RAF-DB Test set images: {len(raf_test)}")

FER2013 Training set images: 28709
FER2013 Test set images: 7178
RAF-DB Test set images: 3068


# 3. Stage 1: End-to-End Training

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimiser, device):
    model.train()

    return epoch_loss, epoch_acc

In [ ]:
def validate(model, data_loader, criterion, device):
    model.eval()

    return val_loss, val_acc

In [ ]:
PATH = "model_checkpoints/latest.pth"
os.makedirs("model_checkpoints", exist_ok=True)

model = SwinXception(num_classes=7).to(device)

optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser)

start_epoch = 0

if os.path.exists(PATH):
    checkpoint = torch.load(PATH, map_location=device)
    start_epoch = checkpoint["epoch"] + 1
    print(f"Checkpoint found! Starting from epoch {start_epoch}...")
    
    model.load_state_dict(checkpoint["model_state_dict"])
    optimiser.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_"])
else:
    print("No model checkpoints found. Starting from epoch 1...")

epochs = 50

for epoch in range(start_epoch, epochs):
    print(f"Epoch {epoch}/{epochs}")
    
    epoch_loss, epoch_acc = train_one_epoch(model, train_loader, criterion, optimiser, device)

    scheduler.step()

    fer_loss, fer_acc = validate(model, test_loader, criterion, device)
    raf_loss, raf_acc = validate(model, test_loader, criterion, device)

# 4. Stage 2: Feature Extraction and Synthesis

# 5. Stage 3: Retraining the MLP Head

# 6. Evaluation Metrics and Visualisation

# 7. Grad-CAM